My goal is to train a `LLM` on Nepali Language dataset from scratch.

## **Data Prep**

We are preparing the dataset from the below sources:

**Wikipedia Dumps**

**Scrapping Sites**

**`Oscar` Hugging Face Datasets**

<hr>

## **Problems to Tackle**

### **Dataset**

We need at least `50` to `100` GB of text data in the target language for effective training.

### **Tokenization**

We cannot use `Tokenizers` that are pre-trained on English text. We need to train our own tokenizers on the target language corpus.

We need to use `BPE` or `Unigram` tokenization methods.

**Training Custom Tokenizers**

- Use `SentencePiece` or `Byte-Pair Encoding (BPE)` to train tokenizers on the collected corpus.

**Target Vocabulary Size**

- A vocab size of `32,000` to `50,000` tokens is generally effective for most languages.

### **Cleaning**

- Normalize Unicode and remove HTML.

- Run MinHash to remove repetitive data.

### **Storage**

- Use efficient storage formats like `TFRecord` or `WebDataset` for large-scale datasets. Or `Parquet` files for easier data manipulation.

<hr>
<hr>



Our first target will be train a `Base Language Model`

## **Base Language Model**

Before everything, we will need a `Base Language Model` trained on a large corpus of text in the target language. This model will serve as the foundation for further fine-tuning and adaptation to specific tasks.

This model will learn `Nepali Grammar`, `Syntax`, and `Vocabulary`. 

This would require a substantial amount of computational resources and time.

**Objective** : It will predict the next word in a sequence given the previous words.

<hr>

Once our base model is trained, we can move on to fine-tuning it for specific tasks.

For example, we can fine-tune it for `Text Classification`, `Named Entity Recognition (NER)`, or `Question Answering`.

Training a GPT-2 model from scratch on 50GB of Nepali text is a significant undertaking. Even though you are using the GPT-2 architecture, training from scratch means the model starts with completely random weights and must learn everything—from the Devanagari alphabet to complex Nepali sentence structures—on its own.

Here are the exact, granular steps to train your base Nepali LLM.

### Phase 1: Custom Tokenizer Training
Before the model can "read," you must build a dictionary that translates Nepali characters into numbers (tokens).

1.  **Select a Subset for Training:** You don't need all 50GB to train a tokenizer. Extract a representative 2–3GB of diverse text.
2.  **Define Vocabulary Size:** For Nepali, a vocabulary size between **32,000 and 50,257** is standard. 
    *   *Reason:* Too small, and the model struggles with complex words; too large, and the model's "brain" is wasted on rare words.
3.  **Choose Algorithm:** Use **Byte-Pair Encoding (BPE)**.
    *   *Reason:* This is what GPT-2 uses. it ensures that even "out-of-vocabulary" words can be broken down into smaller pieces rather than resulting in an "Unknown" error.
4.  **Include Special Tokens:** Explicitly add tokens like `<|endoftext|>` (to mark the end of a document) and padding tokens.
5.  **Train and Save:** Save the `vocab.json` and `merges.txt` files.
    *   *Reason:* These files are the "key" to translating text into data the model understands.

---

### Phase 2: Data Tokenization & Serialization (The "Pre-baking" Phase)
Feeding raw text into a GPU is too slow. We must convert the 50GB of text into a binary format that the GPU can ingest instantly.

1.  **Document Concatenation:** Join all your text files, separated by the `<|endoftext|>` token.
2.  **Mapping Text to IDs:** Run your 50GB corpus through the tokenizer created in Phase 1 to convert every word into an integer (ID).
3.  **Chunking:** Divide the massive stream of integers into fixed-length blocks (usually **1,024 tokens** for GPT-2).
4.  **Storage Format:** Save these tokenized IDs into **Memory-Mapped (memmap) binary files** or **Parquet/Arrow** format.
    *   *Reason:* This allows the training script to read data directly from the hard drive without loading the full 50GB into RAM, preventing "Out of Memory" crashes.

---

### Phase 3: Model Configuration (The Architectural Blueprint)
Since you are training from scratch, you aren't loading weights; you are just defining the shape of the model.

1.  **Define `config.json`:** Set the parameters for GPT-2 (Small, Medium, or Large).
    *   `n_layer`: 12 (for GPT-2 small)
    *   `n_head`: 12
    *   `n_embd`: 768
    *   `block_size`: 1024 (maximum context window)
2.  **Initialize Weights:** The system will initialize the model with **random numbers** (Gaussian distribution).
3.  **Tie Weights:** Ensure the input embedding layer and the final output layer share the same weights.
    *   *Reason:* This reduces the number of parameters the model needs to learn and improves performance in low-resource languages like Nepali.

---

### Phase 4: Training Setup & Hyperparameters
This is where you define "how" the model learns.

1.  **Set Learning Rate (LR):** Start with a high learning rate (e.g., 6e-4) and use a **Learning Rate Scheduler** (Cosine Decay).
    *   *Reason:* The model starts by making big "leaps" in logic and then "fine-tunes" its understanding as training progresses.
2.  **Weight Decay:** Set to 0.1 to prevent the model from memorizing specific sentences (overfitting).
3.  **Batch Size:** Use a large global batch size (e.g., 0.5 million tokens per step). 
    *   *Reason:* Since you have 50GB of data, a large batch size helps the model see enough variety in every update step to stabilize its learning of Nepali grammar.
4.  **Precision Level:** Use **BF16 (BFloat16)** or **FP16 Mixed Precision**.
    *   *Reason:* It speeds up training by 2x–3x and uses half the GPU memory without losing model quality.

---

### Phase 5: The Training Loop (Execution)
The actual process of the model "reading" the data.

1.  **Gradient Accumulation:** If your GPU memory is small, perform several "mini-steps" before updating the model's brain once.
2.  **The Forward Pass:** The model takes 1,023 tokens and tries to predict the 1,024th token.
3.  **Loss Calculation:** The model compares its guess to the actual Nepali word. The difference is the "Loss" (Cross-Entropy Loss).
4.  **The Backward Pass (Backpropagation):** The model calculates how to change its internal connections (weights) to make the Loss smaller next time.
5.  **Optimization:** The **AdamW Optimizer** updates the weights.
6.  **Checkpointing:** Save the model weights every 5,000 steps.
    *   *Reason:* If the power goes out or the system crashes, you don't lose weeks of work.

---

### Phase 6: Monitoring & Quality Control
How you know the model is actually learning Nepali.

1.  **Loss Tracking:** Watch the loss curve. It should drop sharply at first and then level off.
2.  **Perplexity (PPL):** This is the main metric for base models. 
    *   *Reason:* Lower perplexity means the model is becoming "less surprised" by the next word in a Nepali sentence.
3.  **Validation Split:** Keep 1% of your data aside (which the model never sees during training). Regularly test the model on this 1% to ensure it isn't just memorizing.
4.  **Sample Generation:** Every few hours, ask the model to generate a sentence starting with *"नेपाल..."*. 
    *   *Success:* It produces grammatically correct, though perhaps factualy weird, Nepali.
    *   *Failure:* It produces repetitive characters or gibberish.

---

### Summary of Data/File Formats used:
*   **Input Text:** `.txt` or `.jsonl` (UTF-8 encoded).
*   **Tokenizer:** `.json` and `.txt` (Mapping of strings to IDs).
*   **Training Data:** `.bin` or `.npy` (Serialized integers for speed).
*   **Model Weights:** `.bin` or `.safetensors` (The actual "brain" of the model).
*   **Logs:** TensorBoard or WandB files (To visualize the learning progress).